# 01 — Kombinasi Teknik Terbaik: Fitur Tambahan + DART + Sample Weighting/Resampling

Menggabungkan hanya teknik yang **terbukti membantu** di
[experiments/2026-07-09/](../2026-07-09/) (setelah perbaikan split
group-aware) — bukan menumpuk kelima eksperimen kemarin mentah-mentah:

| Teknik | Dipakai? | Alasan |
|---|---|---|
| 23 fitur tambahan (nb04) | **Ya** | Perbaikan kecil tapi konsisten (F1 macro +0.011, F1 weighted +0.002) |
| sample_weight (nb05) | **Ya** | Komponen dari kandidat terbaik & paling robust hari itu |
| DART (nb05) | **Ya** | Komponen dari kandidat terbaik & paling robust hari itu |
| RandomOverSampler (nb02) | **Dicoba sebagai pembanding** | Solid juga (F1 weighted 0.486), dicoba gantikan sample_weight untuk lihat mana yang lebih baik dikombinasikan dengan DART |
| Hyperparameter tuning (nb01) | **Dicoba dengan catatan** | Params tuning XGBoost lumayan robust (acc 52.1%), tapi ditemukan di ruang pencarian `gbtree` + 164 fitur — dipakai ulang di sini sebagai pendekatan, BUKAN re-tuning baru untuk kombinasi 187 fitur + DART |
| SMOTE (nb02) | **Tidak** | Konsisten merugikan di kedua split (sebelum & sesudah perbaikan) |
| GOSS (nb05) | **Tidak** | Konsisten merugikan LightGBM di kedua split |
| Two-stage cascade (nb03) | **Tidak** | Keunggulannya menyusut banyak setelah split diperbaiki; menambah kompleksitas untuk gain kecil, sulit digabung bersih dengan fitur tambahan |

Semua varian dievaluasi di **val set** (aturan main sama seperti
eksperimen 2026-07-09 — lihat [README.md](README.md)), dengan split
group-aware yang sama (`data/processed/`, sudah diperbaiki 2026-07-10).


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser import LINParser
from src.features.engineer import (
    extract_features, SUITS, SEATS,
    hand_hcp_total, hand_is_balanced, hand_void_count,
    hand_singleton_count, hand_doubleton_count,
)
from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-10" / "outputs" / "combined_best_techniques"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Fitur tambahan (23 kolom, sama seperti notebook 04 tanggal 2026-07-09)

In [2]:
SEATS_ORDER = ["N", "E", "S", "W"]


def suit_quick_tricks(cards: list[str]) -> float:
    s = set(cards)
    if "A" in s and "K" in s:
        return 2.0
    if "A" in s and "Q" in s:
        return 1.5
    if "A" in s:
        return 1.0
    if "K" in s and "Q" in s:
        return 1.0
    if "K" in s and len(cards) >= 2:
        return 0.5
    return 0.0


def hand_quick_tricks(hand) -> float:
    return sum(suit_quick_tricks(cards) for cards in hand.as_dict().values())


def hand_total_points(hand) -> int:
    hcp = hand_hcp_total(hand)
    shortness = (
        3 * hand_void_count(hand)
        + 2 * hand_singleton_count(hand)
        + 1 * hand_doubleton_count(hand)
    )
    return hcp + shortness


def suit_quality(cards: list[str]) -> int:
    return sum(1 for c in cards if c in ("A", "K", "Q", "J", "T"))


def hand_suit_quality_best(hand) -> int:
    lengths = {s: len(cards) for s, cards in hand.as_dict().items()}
    best_suit = max(lengths, key=lambda s: (lengths[s], SUITS.index(s)))
    return suit_quality(hand.as_dict()[best_suit])


def second_best_fit(h1, h2) -> int:
    combined = {s: len(h1.as_dict()[s]) + len(h2.as_dict()[s]) for s in SUITS}
    ordered = sorted(combined.values(), reverse=True)
    return ordered[1] if len(ordered) > 1 else 0


def get_opener_seat(board) -> str | None:
    if not board.dealer or board.dealer not in SEATS_ORDER:
        return None
    dealer_idx = SEATS_ORDER.index(board.dealer)
    for i, bid in enumerate(board.auction):
        b = bid.upper().strip()
        if b not in ("P", "PASS", "AP"):
            return SEATS_ORDER[(dealer_idx + i) % 4]
    return None


def extract_advanced_features(board, base_row: dict) -> dict:
    f = {}
    for seat in SEATS:
        hand = board.hands[seat]
        f[f"{seat}_quick_tricks"] = hand_quick_tricks(hand)
        f[f"{seat}_total_points"] = hand_total_points(hand)
        f[f"{seat}_suit_quality_best"] = hand_suit_quality_best(hand)

    for prefix, (s1, s2) in {"ns_": ("N", "S"), "ew_": ("E", "W")}.items():
        h1, h2 = board.hands[s1], board.hands[s2]
        f[f"{prefix}quick_tricks"] = hand_quick_tricks(h1) + hand_quick_tricks(h2)
        f[f"{prefix}total_points"] = hand_total_points(h1) + hand_total_points(h2)
        f[f"{prefix}second_fit"] = second_best_fit(h1, h2)
        best_fit_key = f"{prefix}best_fit"
        f[f"{prefix}lott_level"] = base_row[best_fit_key] - 6

    opener_seat = get_opener_seat(board)
    if opener_seat is not None:
        opener_hand = board.hands[opener_seat]
        f["opener_hcp"] = hand_hcp_total(opener_hand)
        f["opener_balanced"] = int(hand_is_balanced(opener_hand))
    else:
        f["opener_hcp"] = 0
        f["opener_balanced"] = 0

    f["preemptive_opening"] = int(
        base_row.get("opening_level", 0) >= 2 and f["opener_hcp"] < 10
    )

    f["_source_file"] = board.source_file
    f["_room"] = board.room
    f["_board_number"] = board.board_number
    return f


## 2. Parse ulang & join ke split kanonik (group-aware, sudah diperbaiki 2026-07-10)

In [3]:
t0 = time.time()
parser = LINParser()
boards = parser.parse_directory(PROJECT_ROOT / "data" / "raw")
print(f"Parsed {len(boards)} boards in {time.time() - t0:.1f}s")

new_rows = []
for b in boards:
    base_row = extract_features(b)
    if base_row is None:
        continue
    new_rows.append(extract_advanced_features(b, base_row))

df_new = pd.DataFrame(new_rows)
new_feature_cols = [c for c in df_new.columns if not c.startswith("_")]
print(f"New feature columns ({len(new_feature_cols)}): {new_feature_cols}")

df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed")

JOIN_KEYS = ["_source_file", "_room", "_board_number"]


def augment(df_split: pd.DataFrame) -> pd.DataFrame:
    before = len(df_split)
    merged = df_split.merge(df_new, on=JOIN_KEYS, how="left", validate="one_to_one")
    assert len(merged) == before, "Join changed row count"
    assert merged[new_feature_cols].isnull().sum().sum() == 0, "Some rows didn't find a match"
    return merged


df_train_aug = augment(df_train)
df_val_aug = augment(df_val)

all_feature_cols = feature_cols + new_feature_cols
print(f"Total features: {len(all_feature_cols)} (164 kanonik + {len(new_feature_cols)} baru)")

X_train_164 = df_train_aug[feature_cols].values
X_val_164 = df_val_aug[feature_cols].values
X_train_187 = df_train_aug[all_feature_cols].values
X_val_187 = df_val_aug[all_feature_cols].values
y_train = df_train_aug["label"].values
y_val = df_val_aug["label"].values

sw_train = compute_sample_weight("balanced", y_train)


Parsed 11236 boards in 8.8s
New feature columns (23): ['N_quick_tricks', 'N_total_points', 'N_suit_quality_best', 'E_quick_tricks', 'E_total_points', 'E_suit_quality_best', 'S_quick_tricks', 'S_total_points', 'S_suit_quality_best', 'W_quick_tricks', 'W_total_points', 'W_suit_quality_best', 'ns_quick_tricks', 'ns_total_points', 'ns_second_fit', 'ns_lott_level', 'ew_quick_tricks', 'ew_total_points', 'ew_second_fit', 'ew_lott_level', 'opener_hcp', 'opener_balanced', 'preemptive_opening']
Total features: 187 (164 kanonik + 23 baru)


## 3. Anchor pembanding (hasil 2026-07-09, dihitung ulang di sini untuk memastikan
kondisi identik — sama split, sama kernel, sama waktu eksekusi)

In [4]:
BASE_XGB = dict(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)
DART_PARAMS = dict(booster="dart", rate_drop=0.1, skip_drop=0.5, normalize_type="tree")

anchors = []

# Anchor A: baseline murni, 164 fitur
t0 = time.time()
clf = XGBClassifier(**BASE_XGB)
clf.fit(X_train_164, y_train)
res = evaluate(y_val, clf.predict(X_val_164), clf.predict_proba(X_val_164), le,
                model_name="ANCHOR: XGBoost baseline (164 fitur)")
res["fit_seconds"] = round(time.time() - t0, 1)
anchors.append(res)
print_summary(res)

# Anchor B: sample_weight + DART, 164 fitur (kandidat terbaik 2026-07-09)
t0 = time.time()
clf = XGBClassifier(**{**BASE_XGB, **DART_PARAMS})
clf.fit(X_train_164, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val_164), clf.predict_proba(X_val_164), le,
                model_name="ANCHOR: XGBoost +sample_weight+dart (164 fitur)")
res["fit_seconds"] = round(time.time() - t0, 1)
anchors.append(res)
print_summary(res)



  ANCHOR: XGBoost baseline (164 fitur)
  Accuracy          : 0.5127
  Precision (macro) : 0.3335
  Recall (macro)    : 0.2515
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4685
  Top-3 Accuracy    : 0.7508
  Top-5 Accuracy    : 0.8480

  ANCHOR: XGBoost +sample_weight+dart (164 fitur)
  Accuracy          : 0.4918
  Precision (macro) : 0.3156
  Recall (macro)    : 0.3455
  F1 (macro)        : 0.3217
  F1 (weighted)     : 0.4983
  Top-3 Accuracy    : 0.7482
  Top-5 Accuracy    : 0.8330


## 4. Varian gabungan

In [5]:
combined_results = []

# Varian 1: 187 fitur + sample_weight + DART
t0 = time.time()
clf = XGBClassifier(**{**BASE_XGB, **DART_PARAMS})
clf.fit(X_train_187, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val_187), clf.predict_proba(X_val_187), le,
                model_name="V1: 187 fitur + sample_weight + dart")
res["fit_seconds"] = round(time.time() - t0, 1)
combined_results.append(res)
print_summary(res)

# Varian 2: 187 fitur + RandomOverSampler + DART (ganti mekanisme weighting)
ros = RandomOverSampler(random_state=42)
X_train_187_ros, y_train_ros = ros.fit_resample(X_train_187, y_train)
t0 = time.time()
clf = XGBClassifier(**{**BASE_XGB, **DART_PARAMS})
clf.fit(X_train_187_ros, y_train_ros)
res = evaluate(y_val, clf.predict(X_val_187), clf.predict_proba(X_val_187), le,
                model_name="V2: 187 fitur + RandomOverSampler + dart")
res["fit_seconds"] = round(time.time() - t0, 1)
res["n_train_rows"] = int(len(y_train_ros))
combined_results.append(res)
print_summary(res)

# Varian 3: 187 fitur + sample_weight + DART + hyperparameter hasil tuning nb01
# (tuned di ruang pencarian gbtree/164 fitur — dipakai ulang sebagai pendekatan,
# BUKAN re-tuning baru untuk kombinasi 187 fitur + DART. Base params diganti,
# override tetap booster=dart + rate_drop/skip_drop dari DART_PARAMS.)
tuned_summary = json.loads(
    (PROJECT_ROOT / "experiments/2026-07-09/outputs/hyperparameter_tuning/summary.json").read_text()
)
tuned_xgb_params = tuned_summary["best_params"]["XGBoost"]
print("Params tuning nb01 dipakai ulang:", tuned_xgb_params)

V3_PARAMS = dict(
    random_state=42, n_jobs=-1, eval_metric="mlogloss", verbosity=0,
    **tuned_xgb_params, **DART_PARAMS,
)
t0 = time.time()
clf = XGBClassifier(**V3_PARAMS)
clf.fit(X_train_187, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val_187), clf.predict_proba(X_val_187), le,
                model_name="V3: 187 fitur + sample_weight + dart + tuned params (nb01)")
res["fit_seconds"] = round(time.time() - t0, 1)
combined_results.append(res)
print_summary(res)



  V1: 187 fitur + sample_weight + dart
  Accuracy          : 0.4866
  Precision (macro) : 0.3179
  Recall (macro)    : 0.3344
  F1 (macro)        : 0.3153
  F1 (weighted)     : 0.4932
  Top-3 Accuracy    : 0.7410
  Top-5 Accuracy    : 0.8369

  V2: 187 fitur + RandomOverSampler + dart
  Accuracy          : 0.4925
  Precision (macro) : 0.3235
  Recall (macro)    : 0.3152
  F1 (macro)        : 0.3068
  F1 (weighted)     : 0.4940
  Top-3 Accuracy    : 0.7384
  Top-5 Accuracy    : 0.8271
Params tuning nb01 dipakai ulang: {'subsample': 1.0, 'reg_lambda': 2.0, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.08, 'colsample_bytree': 1.0}

  V3: 187 fitur + sample_weight + dart + tuned params (nb01)
  Accuracy          : 0.4690
  Precision (macro) : 0.3096
  Recall (macro)    : 0.3357
  F1 (macro)        : 0.3127
  F1 (weighted)     : 0.4837
  Top-3 Accuracy    : 0.7378
  Top-5 Accuracy    : 0.8330


## 5. Bandingkan semua (anchor + varian gabungan)

In [6]:
all_results = anchors + combined_results
comparison = compare_models(all_results)
comparison


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
ANCHOR: XGBoost baseline (164 fitur),0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
ANCHOR: XGBoost +sample_weight+dart (164 fitur),0.491846,0.315552,0.345488,0.321666,0.498336,0.748206,0.833007
V1: 187 fitur + sample_weight + dart,0.486628,0.317939,0.334397,0.315322,0.493236,0.741031,0.836921
V2: 187 fitur + RandomOverSampler + dart,0.492498,0.323455,0.315176,0.306757,0.494004,0.738421,0.827136
V3: 187 fitur + sample_weight + dart + tuned params (nb01),0.469015,0.309644,0.335656,0.312709,0.483737,0.737769,0.833007


In [ ]:
comparison.to_csv(OUT_DIR / "val_comparison.csv")

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "new_feature_cols": new_feature_cols,
    "reused_tuned_params_source": "experiments/2026-07-09/outputs/hyperparameter_tuning/summary.json",
    "results": {r["model"]: {k: r.get(k) for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy", "fit_seconds"]}
        for r in all_results},
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(f"Saved: {OUT_DIR / 'val_comparison.csv'}")
print(f"Saved: {OUT_DIR / 'summary.json'}")


Saved: d:\Bridge-Prediction\experiments\2026-07-10\outputs\combined_best_techniques\val_comparison.csv
Saved: d:\Bridge-Prediction\experiments\2026-07-10\outputs\combined_best_techniques\summary.json


: 

## 6. Kesimpulan

Hasil val set (`experiments/2026-07-10/outputs/combined_best_techniques/val_comparison.csv`):

| Model | Accuracy | F1 macro | F1 weighted |
|---|---|---|---|
| **ANCHOR: 164 fitur + sample_weight + DART** (kandidat terbaik 2026-07-09) | 49.2% | **0.322** | **0.498** |
| V2: 187 fitur + RandomOverSampler + DART | 49.2% | 0.307 | 0.494 |
| V1: 187 fitur + sample_weight + DART | 48.7% | 0.315 | 0.493 |
| V3: 187 fitur + sample_weight + DART + tuned params (nb01) | 46.9% | 0.313 | 0.484 |
| ANCHOR: XGBoost baseline (164 fitur) | 51.3% | 0.269 | 0.469 |

**Jawaban untuk pertanyaan awal ("apakah perlu digabungkan semua
metode"): TIDAK, dan menggabungkan yang "terbukti membantu" pun di sini
JUSTRU SEDIKIT MENURUNKAN hasil, bukan menaikkan.**

- **Tidak satu pun varian gabungan (V1/V2/V3) mengalahkan anchor
  tunggal** (164 fitur + sample_weight + DART, F1 macro 0.322 / F1
  weighted 0.498) yang sudah ditemukan tanggal 2026-07-09. Menambahkan
  23 fitur baru ke kombinasi yang sudah bagus ini malah menurunkan F1
  macro (0.322→0.315 di V1) — berlawanan dengan efeknya saat
  ditambahkan ke XGBoost polos di notebook 04 (2026-07-09), yang
  menaikkan F1 macro (+0.011).
- **Interpretasi**: perbaikan dari teknik-teknik ini TIDAK bersifat
  aditif/independen. 23 fitur baru itu sebagian besar kombinasi
  linear dari fitur yang sudah ada (`total_points = hcp+shortness`,
  `lott_level = best_fit-6`) — DART dengan dropout-nya sendiri sudah
  "menemukan" kombinasi semacam itu secara implisit lewat tree yang
  berbeda-beda di tiap round; menambah kolom yang sebagian redundan
  hanya menambah noise pilihan split tanpa menambah informasi baru
  untuk DART, meski itu MEMBANTU model biasa yang tidak punya mekanisme
  dropout ini.
- **RandomOverSampler + DART (V2) tidak lebih baik dari sample_weight +
  DART** — konsisten dengan temuan bahwa oversampling 7x lipat data
  menambah beban komputasi (dataset lebih besar) tanpa manfaat yang
  sepadan ketika DART sudah menangani imbalance lewat sample_weight.
- **Menambahkan params tuning nb01 (V3) ke DART adalah yang paling
  buruk** dari ketiganya (F1 weighted 0.484) — masuk akal karena params
  itu ditemukan untuk `booster="gbtree"` polos (bukan `dart`), jadi
  tidak otomatis optimal ketika booster-nya diganti. Re-tuning khusus
  untuk kombinasi DART (bukan reuse params gbtree) diperlukan kalau mau
  menguji ini dengan benar.
- **Pelajaran metodologis**: dua peningkatan yang masing-masing
  terbukti valid secara terpisah **tidak menjamin** saling menambah
  ketika digabung — perlu selalu diuji ulang sebagai kombinasi, bukan
  diasumsikan aditif. Kandidat terbaik keseluruhan (dari kedua hari)
  tetap **164 fitur + sample_weight + DART murni**, tanpa fitur
  tambahan maupun resampling maupun tuning tambahan.
- **Rekomendasi**: hentikan upaya menggabungkan teknik lebih lanjut
  untuk kombinasi ini secara spesifik; validasi kandidat 164 fitur +
  sample_weight + DART di **test set** sebagai langkah berikutnya yang
  paling bernilai, alih-alih mencoba kombinasi baru tanpa arah yang
  jelas.
